In [1]:
%%capture
!pip install gradio
!pip install transformers
!pip install torch
!pip install pypdf

##**FaceBook Bart - Large - CNN**

BART is a transformer encoder-encoder (seq2seq) model with a bidirectional (BERT-like) encoder and an autoregressive (GPT-like) decoder. BART is pre-trained by corrupting text with an arbitrary noising function, and learning a model to reconstruct the original text.

BART is particularly effective when fine-tuned for text generation (e.g. summarization, translation) but also works well for comprehension tasks (e.g. text classification, question answering). This particular checkpoint has been fine-tuned on CNN Daily Mail, a large collection of text-summary pairs.

In [6]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from pypdf import PdfReader

In [3]:
def extract_text_from_pdf(pdf_file):
    '''
    Opens a PDF file using PdfReader, iterates through each page,
    extracts the text content in plain text format, and returns the
    concatenated text as a single string.

    Args:
      pdf_file -> pdf file's route which in our case will be its name in colab enviroment
    '''

    reader = PdfReader(pdf_file)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

##Load the model, tockenizer, and pipeline from hugginface

In [7]:
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [10]:
def summarize_report(pdf_file):
    '''
    Summarizes the content of a PDF file.

    Args:
        pdf_file (str): The path to the PDF file.

    Returns:
        str: The generated summary of the PDF content.
    '''
    # Extract text from the PDF file
    text = extract_text_from_pdf(pdf_file)

    # Tokenize the extracted text, returning PyTorch tensors
    # Truncate the text to a maximum length of 1024 tokens
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)

    # Generate the summary using the loaded model
    # Set maximum and minimum length for the summary
    # Apply a length penalty to discourage overly short summaries
    # Use beam search with 4 beams for better summary generation
    summary_ids = model.generate(inputs.input_ids,
                                max_length=1024,
                                min_length=50,
                                length_penalty=2.0,
                                num_beams=4
                                )

    # Decode the generated summary IDs back to text
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # Return the generated summary
    return summary

##Gradio, Simple Chatbot Interface for our Customer Service

**Gradio** is a Python library that allows you to quickly create easy-to-use **web interfaces** for machine learning models or Python functions. It’s commonly used to demo, test, or share models interactively without writing complex frontend code. In your example, `gr.Interface()` wraps the `answer_customer_question` function in a simple web app where users can type a question (`inputs="text"`) and see the model’s response (`outputs="text"`). The `title` sets the interface name (“Customer Support Chatbot”), and `iface.launch()` starts a local web server, opening the interface in a browser so anyone can interact with your chatbot in real time.


In [11]:
iface = gr.Interface(
      fn=summarize_report,
      inputs = gr.File(label="Upload PDF File"),
      outputs = gr.Textbox(label="Generated Summary"),
      title="AI PDF Report Summarizer",
      description="Upload a PDF report, and AI will summarize it by extracting key insights"
)
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://69c8dcb6b6b9022c85.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
